# 🚀 VS#-15M Deep Neural Language Model Trainer (Kaggle T4/P100 GPU)
### Continuous Adaptive Fine-Tuning: Dynamic Epoch Cycles, Warm-Start & Auto HF Sync

This notebook is an updated, optimized trainer for **VS#-15M** (15 Million Parameter VerScript neural model).

**Key Upgrades in this runner:**
- Seamless support for both raw (`model_weights.json`) and compressed (`model_weights.json.gz`) weights.
- Automatic ingestion of Kaggle Secrets (`HF_TOKEN`) with fallback to prompt.
- Anti-memorization pipeline: AdamW weight decay, label smoothing, and dropout.
- Mixed-Precision FP16 acceleration for 2x faster Tensor Core throughput on T4/P100.
- Interactive prompt generation tester with temperature and top-p sampling.
- Checkpoint staging to `/kaggle/working` for direct 1-click browser download.


## 1. Verify GPU Setup


In [ ]:
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi


## 2. Clone Clean VS-Sharp Repository & Pull Latest Hugging Face Checkpoint


In [ ]:
import os, urllib.request
%cd /kaggle/working
!rm -rf VS-Sharp
!git clone https://github.com/VerScript/VS-Sharp.git
%cd /kaggle/working/VS-Sharp

HF_URLS = [
    "https://huggingface.co/alexon-v8/VS-Sharp/resolve/main/model_weights.json.gz",
    "https://huggingface.co/alexon-v8/VS-Sharp/resolve/main/model_weights.json"
]

downloaded = False
for url in HF_URLS:
    try:
        fname = url.split("/")[-1]
        print(f"[*] Attempting download from {url}...")
        urllib.request.urlretrieve(url, fname)
        if os.path.exists(fname) and os.path.getsize(fname) > 1000:
            mb = os.path.getsize(fname) / (1024 * 1024)
            print(f"✅ Successfully downloaded {fname} ({mb:.1f} MB)!")
            downloaded = True
            break
    except Exception as e:
        print(f"[-] Could not fetch from {url}: {e}")

if not downloaded:
    print("⚠️ No checkpoint found; training will initialize fresh weights or use local knowledge base.")


## 3. Rebuild Master Knowledge Dataset


In [ ]:
!node build_dataset.js


## 4. Run Training with Mixed Precision, Regularization & Hugging Face Push


In [ ]:
import os, getpass, subprocess

# Retrieve Hugging Face token from Kaggle Secrets, environment, or prompt
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    print("💡 Tip: Add HF_TOKEN under Add-ons > Secrets in Kaggle for zero-prompt auto-sync.")
    hf_token = getpass.getpass("Enter Hugging Face Write Token (or press Enter to train locally without push): ").strip()

!pip install -q huggingface_hub

cmd = [
    "python", "train_gpu.py",
    "--epochs", "50",
    "--batch-size", "128",
    "--samples-per-epoch", "20000",
    "--lr", "0.00015",
    "--dropout", "0.15",
    "--label-smoothing", "0.1",
    "--weight-decay", "0.01",
    "--hf-repo", "alexon-v8/VS-Sharp"
]

if hf_token:
    cmd.extend(["--hf-token", hf_token])

subprocess.run(cmd)


## 5. Loss & Accuracy Curves


In [ ]:
import json, os
import matplotlib.pyplot as plt

hist_file = "epoch_eval_history.json"
if os.path.exists(hist_file):
    with open(hist_file, "r", encoding="utf-8") as f:
        history = json.load(f)

    epochs = [h["epoch"] for h in history]
    losses = [h["loss"] for h in history]
    accuracies = [h["accuracy"] * 100 for h in history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(epochs, losses, color="#BD00FF", linewidth=2, marker="o", markersize=3)
    ax1.set_title("Cross-Entropy Loss", fontweight="bold")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.grid(True, linestyle="--", alpha=0.5)

    ax2.plot(epochs, accuracies, color="#00FFCC", linewidth=2, marker="s", markersize=3)
    ax2.set_title("Next-Token Prediction Accuracy (%)", fontweight="bold")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy (%)")
    ax2.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()
else:
    print("No history file found yet.")


## 6. Live Interactive VerScript Inference Test


In [ ]:
import torch, json, gzip, os
from train_gpu import VSSharp15M, generate_text

target_weights = "model_weights.json"
if not os.path.exists(target_weights) and os.path.exists("model_weights.json.gz"):
    with gzip.open("model_weights.json.gz", "rt", encoding="utf-8") as gz:
        w_data = json.load(gz)
elif os.path.exists(target_weights):
    with open(target_weights, "r", encoding="utf-8") as f:
        w_data = json.load(f)
else:
    w_data = None

if w_data:
    vocab = w_data["vocab"]
    vocab_map = {t: idx for idx, t in enumerate(vocab)}
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = VSSharp15M(
        vocab_size=len(vocab),
        embed_dim=256,
        hidden_size_1=w_data["hyperparameters"]["hidden_size_1"],
        hidden_size_2=512
    ).to(device)
    model.eval()

    test_prompts = [
        "User: How do I declare a function in VerScript?\nAssistant:",
        "User: Draw a rectangle and circle in VerScript\nAssistant:",
        "User: Explain terminal and console subsystem in VerScript\nAssistant:"
    ]

    for p in test_prompts:
        print("=" * 60)
        print(p)
        ans = generate_text(model, vocab, vocab_map, p, max_tokens=60, temperature=0.7, top_p=0.9, device=device)
        print(ans)


## 7. Stage Outputs for 1-Click Kaggle Download


In [ ]:
!cp /kaggle/working/VS-Sharp/model_weights.json.gz /kaggle/working/ 2>/dev/null || true
!cp /kaggle/working/VS-Sharp/epoch_eval_history.json /kaggle/working/ 2>/dev/null || true
print("✅ Checkpoints staged in /kaggle/working/")
print("📥 Download directly from Kaggle Right Sidebar: Data -> Output")
